In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm
import yfinance as yf
import pandas_datareader.data as web
import datetime
import requests
import seaborn as sns

In [109]:
from scipy.stats import norm
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [4]:
merged_data = pd.read_csv('merged_data.csv')

In [7]:
merged_data.head()

,Unnamed: 0,date,underlying_price,implied_volatility,risk_free_rate
0,0,2024-04-30,501.98,0.1565,5.25
1,1,2024-05-01,500.35,0.1539,5.26
2,2,2024-05-02,505.03,0.1468,5.25
3,3,2024-05-03,511.29,0.1349,5.25
4,4,2024-05-06,516.57,0.1349,5.25


In [9]:
merged_data.describe()

,Unnamed: 0,underlying_price,implied_volatility,risk_free_rate
count,253.000000,253.000000,253.000000,253.000000
mean,126.000000,565.170474,0.181579,4.630158
std,73.179004,28.127478,0.060426,0.430384
min,0.000000,496.480000,0.118600,4.170000
25%,63.000000,544.510000,0.142800,4.220000
50%,126.000000,563.980000,0.164300,4.480000
75%,189.000000,590.300000,0.199000,5.140000
max,252.000000,612.930000,0.523300,5.260000


In [13]:
merged_data['date'] = pd.to_datetime(merged_data['date'])

In [15]:
merged_data['rate'] = merged_data['risk_free_rate'] / 100

In [17]:
merged_data.head()

,Unnamed: 0,date,underlying_price,implied_volatility,risk_free_rate,rate
0,0,2024-04-30,501.98,0.1565,5.25,0.0525
1,1,2024-05-01,500.35,0.1539,5.26,0.0526
2,2,2024-05-02,505.03,0.1468,5.25,0.0525
3,3,2024-05-03,511.29,0.1349,5.25,0.0525
4,4,2024-05-06,516.57,0.1349,5.25,0.0525


In [111]:
CURVATURE = 0.20    
SKEW      = -0.25   
SKEW_DECAY   = 1.0     
NOISE_STD = 0.005 
OPTION_TYPE = 'call'

In [47]:
# MONEYNESS = np.array([0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20])
# MATURITIES = np.array([21/252, 42/252, 63/252, 126/252, 189/252, 252/252])
 
# np.random.seed(42)

In [113]:
def synthetic_iv(base_iv, E, S, T,
                 curvature=CURVATURE, skew=SKEW,
                 skew_decay=SKEW_DECAY, noise_std=NOISE_STD):

    x              = np.log(E / S)
    curvature_term = curvature * x**2
    skew_term      = skew * x * np.exp(-skew_decay * T)
    noise          = np.random.normal(0, noise_std)
    iv             = base_iv + curvature_term + skew_term + noise
    iv = float(np.clip(iv, 0.03, 1.50))
    return iv 

In [157]:
def synthetic_iv(base_iv, E, S, T, r,
                 curvature0=0.15,
                 curvature1=0.08,
                 curvature_decay=1.0,
                 skew0=-0.22,
                 skew_decay=1.2,
                 atm_term_alpha=0.03,
                 atm_term_decay=1.0,
                 noise_std=0.003,
                 wing_noise_scale=0.75,
                 sigma_floor=0.03,
                 sigma_cap=1.50,
                 rng=None):

    if rng is None:
        rng = np.random.default_rng()

    T = max(T, 1e-10)

   
    F = S * np.exp(r * T)
    x = np.log(E / F)

     
    atm_T = base_iv * (1.0 + atm_term_alpha * np.exp(-atm_term_decay * T))

     
    curvature_T = curvature0 + curvature1 * np.exp(-curvature_decay * T)
    skew_T = skew0 * np.exp(-skew_decay * T)

    
    noise_std_eff = noise_std * (1.0 + wing_noise_scale * abs(x))
    noise = rng.normal(0.0, noise_std_eff)

    iv = atm_T + curvature_T * x**2 + skew_T * x + noise
    iv = float(np.clip(iv, sigma_floor, sigma_cap))
    return iv

In [159]:
def bs_price(S, E, T, r, sigma, option_type=OPTION_TYPE):
    if T < 1e-6 or sigma < 1e-6:
        return max(S - E, 0) if option_type == 'call' else max(E - S, 0)
    d1 = (np.log(S/E) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    if option_type == 'call':
        return S*norm.cdf(d1) - E*np.exp(-r*T)*norm.cdf(d2)
    return E*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

def bs_delta(S, E, T, r, sigma, option_type = OPTION_TYPE):
    if T < 1e-6 or sigma < 1e-6:
        return 1.0 if (option_type == 'call' and S > E) else 0.0
    d1 = (np.log(S/E) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    return norm.cdf(d1) if option_type == 'call' else norm.cdf(d1) - 1

def bs_vega(S, E, T, r, sigma):
    if T < 1e-6 or sigma < 1e-6: return 0.0
    d1 = (np.log(S/E) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T)

def bs_gamma(S, E, T, r, sigma):
    if T < 1e-6 or sigma < 1e-6: return 0.0
    d1 = (np.log(S/E) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    return norm.pdf(d1) / (S * sigma * np.sqrt(T))

In [161]:
def create_synthetic_option_panel(
    merged_data,
    moneyness_grid=np.array([0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20]),
    maturities=np.array([21/252, 42/252, 63/252, 126/252, 189/252, 252/252]),
    issue_every_n_days=5,
    min_days_to_expiry=14, # get rid of data of last 14 days
    seed=42
):
    
    df = merged_data.copy()
    df = df.sort_values("date").reset_index(drop=True)
    df["date"] = pd.to_datetime(df["date"])

    rng = np.random.default_rng(seed)
    records = []

    for issue_idx in range(0, len(df), issue_every_n_days):
        issue_row = df.iloc[issue_idx]
        issue_date = issue_row["date"]
        S_issue = float(issue_row["underlying_price"])

        strikes = S_issue * moneyness_grid

        for T0 in maturities:
            max_days_expiry = int(round(T0 * 252))

            for E in strikes:
                option_id = f"{issue_date.date()}|E={E:.4f}|T0={int(round(T0*252))}"

                for j in range(max_days_expiry + 1):
                    time_elapse_idx = issue_idx + j
                    if time_elapse_idx >= len(df):
                        break

                    tau = T0 - j / 252.0
                    if tau < min_days_to_expiry / 252.0:
                        break

                    row = df.iloc[time_elapse_idx]
                    date = row["date"]
                    S = float(row["underlying_price"])
                    base_iv = float(row["implied_volatility"])
                    r = float(row["rate"])

                   

                    sigma = synthetic_iv(
                        base_iv=base_iv,
                        E=E,
                        S=S,
                        T=tau,
                        r=r,
                        rng=rng
                        )

                    price = bs_price(S, E, tau, r, sigma)
                    delta = bs_delta(S, E, tau, r, sigma)
                    vega = bs_vega(S, E, tau, r, sigma)
                    gamma = bs_gamma(S, E, tau, r, sigma)

                    records.append({
                        "option_id": option_id,
                        "issue_date": issue_date,
                        "date": date,
                        "issue_idx": issue_idx,
                        "time_elapse_idx": time_elapse_idx,
                        "days_since_issue": j,
                        "strike": E,
                        "initial_T": T0,
                        "tau": tau,
                        "underlying_price": S,
                        "rate": r,
                        "implied_vol": sigma,
                        "option_price": price,
                        "delta_bs": delta,
                        "vega_bs": vega,
                        "gamma_bs": gamma
                    })

    panel = pd.DataFrame(records)
    panel = panel.sort_values(["option_id", "date"]).reset_index(drop=True)
    return panel

In [163]:
panel = create_synthetic_option_panel(
    merged_data,
    issue_every_n_days=5
)



In [ ]:
# store dataset in long panel format, one row per contract-date observation

In [164]:
print(panel.shape)
panel.head()

(189009, 16)


,option_id,issue_date,date,issue_idx,time_elapse_idx,days_since_issue,strike,initial_T,tau,underlying_price,rate,implied_vol,option_price,delta_bs,vega_bs,gamma_bs
0,2024-04-30|E=401.5840|T0=126,2024-04-30,2024-04-30,0,0,0,401.584,0.5,0.500000,501.98,0.0525,0.204991,111.915333,0.963515,28.377371,0.001099
1,2024-04-30|E=401.5840|T0=126,2024-04-30,2024-05-01,0,1,1,401.584,0.5,0.496032,500.35,0.0526,0.198617,110.086748,0.966255,26.424310,0.001071
2,2024-04-30|E=401.5840|T0=126,2024-04-30,2024-05-02,0,2,2,401.584,0.5,0.492063,505.03,0.0525,0.194406,114.410043,0.973709,21.596275,0.000885
3,2024-04-30|E=401.5840|T0=126,2024-04-30,2024-05-03,0,3,3,401.584,0.5,0.488095,511.29,0.0525,0.184471,120.267057,0.983712,14.518030,0.000617
4,2024-04-30|E=401.5840|T0=126,2024-04-30,2024-05-06,0,4,4,401.584,0.5,0.484127,516.57,0.0525,0.186088,125.399838,0.986298,12.590771,0.000524
